In [36]:
from pathlib import Path

from imagematerials.__main__ import export_summary_netcdf, simulate_stocks
from imagematerials.vehicles.preprocessing import (
    preprocessing,
)
from imagematerials.util import import_from_netcdf, export_to_netcdf
from imagematerials.model import GenericMainModel, GenericMaterials, GenericStocks
from imagematerials.factory import ModelFactory
from imagematerials.maintenance import Maintenance
from imagematerials.concepts import vehicle_knowledge_graph

import prism


In [37]:
base_dir = "../data/raw"
prep_fp = Path("prep_vema.nc")

In [38]:
if not prep_fp.is_file():
    _, orig_prep_data = preprocessing(base_dir)
    export_to_netcdf(orig_prep_data, prep_fp)
prep_data = import_from_netcdf(prep_fp)
prep_data["weights"] = prep_data.pop("vehicle_weights")

In [ ]:
import pandas as pd
from imagematerials.util import dataset_to_array

# Copy dimensiomns from material_fractions for xr_maintenance_material
materials = prep_data['material_fractions'].coords["material"]
types = prep_data['material_fractions'].coords["Type"]

maintenance_material_pd : pd.DataFrame = pd.read_csv(
    "../data/raw/vehicles/standard_data/all_vehicle_maintenance_image.csv", index_col=0)

stacked_maintenance_material = maintenance_material_pd.set_index("Type").stack().rename_axis(index=["Type", "material"]).reset_index(name="value")

stacked_maintenance_material = stacked_maintenance_material.set_index(["Type", "material"])

stacked_maintenance_material_xr = stacked_maintenance_material.to_xarray()
maintenance_material = dataset_to_array(stacked_maintenance_material_xr, ["Type", "materials"], [])


In [40]:
import numpy as np
import xarray as xr
from scipy.stats import norm

# Define expected value of folded normal distribution
def expected_folded_norm(c, scale):
    term1 = np.sqrt(2 / np.pi) * np.exp(-c**2 / 2)
    term2 = c * (1 - 2 * norm.cdf(-c))
    return scale * (term1 + term2)

# Vectorized function to apply over the last axis (ScipyParam)
def compute_expected_lifetime(params):
    c = params[0]
    scale = params[1]
    return expected_folded_norm(c, scale)

selected_params = prep_data["lifetimes"]["folded_norm"].sel(Time=2020).squeeze()

# Apply over the xarray DataArray
expected_lifetimes = xr.apply_ufunc(
    compute_expected_lifetime,
    selected_params,
    input_core_dims=[["ScipyParam"]],
    output_core_dims=[[]],
    vectorize=True,
    dask="parallelized",  # remove if not using Dask
    output_dtypes=[float]
)

# Now `expected_lifetimes` should have dims: ("Time", "Type")



In [ ]:
maintenance_material_per_year = (maintenance_material / expected_lifetimes).drop("Time")
maintenance_material_per_year_broadcasted = vehicle_knowledge_graph.rebroadcast_xarray_impute(
    maintenance_material_per_year, types.values)

prep_data["maintenance_material_fractions"] = maintenance_material_per_year_broadcasted

C:\Users\5982758\AppData\Local\Temp\ipykernel_6524\2745639944.py:1: DeprecationWarning: dropping variables using `drop` is deprecated; use drop_vars.
  maintenance_material_per_year = (maintenance_material / expected_lifetimes).drop("Time")


<xarray.DataArray (Type: 53, materials: 10)> Size: 4kB
array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
...
        0.00000000e+00, 0.00000000e+00, 5.72944297e-05, 0.00000000e+00,
        2.93587533e-03, 0.00000000e+00],
       [2.48037135e-04, 0.00000000e+00, 9.91114058e-05, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 5.72944297e-05, 0.00000000e+00,
        2.93587533e-03, 0.00000000e+00],
       [2.48037135e-04, 0.00000000e+00, 9.91114058e-05, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 5.72944297e-05, 0.00000000e+00,
        2.93587533e-03, 0.00000000e+00],
       [2.48037135e-04, 0.00000000e+00, 9.91114058e-05, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 5.72944297e-05, 0.00000000e+00,
        2.93587533e-03, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [3.47467025e-03, 0.00000000e+00, 5.95657758e-04, 3.20417850e-04,
        0.00000000e+00, 0.00000000e+00, 1.35942046e-03, 4.53751666e-04,
        5.85730129e-03, 2.31157795e-04],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00]])
Coordinates:
  * Type       (Type) <U35 7kB 'Bikes' 'Cars' ... 'Trains' 'Very Large Ships'
  * materials  (materials) object 80B 'Aluminium' 'Co' 'Cu' ... 'Steel' 'Wood'

In [42]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [43]:
# Define the coordinates of all dimensions.
Region = list(prep_data["stocks"].coords["Region"].values)
Time = [t for t in complete_timeline]
Cohort = Time
Type = list(prep_data["stocks"].coords["Type"].values)
material = list(prep_data["material_fractions"].coords["material"].values)

# Create
main_model_normal = GenericMainModel(
    complete_timeline, Region=Region, Time=Time, Cohort=Cohort, Type=Type, prep_data=prep_data,
    compute_materials=True, compute_battery_materials=False, compute_maintenance_materials=True, 
    material=material)

In [44]:
main_model_normal.simulate(simulation_timeline)

In [45]:
main_model_factory = ModelFactory(
    prep_data, complete_timeline
    ).add(GenericStocks
    ).add(GenericMaterials
    ).add(Maintenance
    ).finish()

In [46]:
main_model_factory.simulate(simulation_timeline)

In [60]:
main_model_normal.maintenance_model.inflow_maintenance[2020]

<xarray.DataArray (Region: 28, Type: 53, materials: 10)> Size: 119kB
array([[[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        ...,
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [8.06454073e+04, 0.00000000e+00, 1.38249270e+04, ...,
         1.05313556e+04, 1.35945115e+05, 5.36506003e+03],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00]],

       [[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
...
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [5.86968381e+05, 0.00000000e+00, 1.00623151e+05, ...,
         7.66512680e+04, 9.89460985e+05, 3.90489763e+04],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00]],

       [[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        ...,
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [5.06703185e+05, 0.00000000e+00, 8.68634031e+04, ...,
         6.61695635e+04, 8.54156797e+05, 3.37092104e+04],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00]]])
Coordinates:
    Time       int64 8B 2020
  * Region     (Region) <U2 224B '1' '10' '11' '12' '13' ... '5' '6' '7' '8' '9'
  * Type       (Type) <U35 7kB 'Bikes' 'Cars' ... 'Trains' 'Very Large Ships'
  * materials  (materials) object 80B 'Aluminium' 'Co' 'Cu' ... 'Steel' 'Wood'

In [64]:
main_model_normal.maintenance_model.inflow_maintenance[2020]

<xarray.DataArray (Region: 28, Type: 53, materials: 10)> Size: 119kB
array([[[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        ...,
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [8.06454073e+04, 0.00000000e+00, 1.38249270e+04, ...,
         1.05313556e+04, 1.35945115e+05, 5.36506003e+03],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00]],

       [[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
...
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [5.86968381e+05, 0.00000000e+00, 1.00623151e+05, ...,
         7.66512680e+04, 9.89460985e+05, 3.90489763e+04],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00]],

       [[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        ...,
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [5.06703185e+05, 0.00000000e+00, 8.68634031e+04, ...,
         6.61695635e+04, 8.54156797e+05, 3.37092104e+04],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00]]])
Coordinates:
    Time       int64 8B 2020
  * Region     (Region) <U2 224B '1' '10' '11' '12' '13' ... '5' '6' '7' '8' '9'
  * Type       (Type) <U35 7kB 'Bikes' 'Cars' ... 'Trains' 'Very Large Ships'
  * materials  (materials) object 80B 'Aluminium' 'Co' 'Cu' ... 'Steel' 'Wood'